CASE STUDY: LOAN PREDICTION – DATA PREPROCESSING



Loading the Dataset

The first step is to load the given loan prediction dataset into the Python environment.
For this, the Pandas library is used because it makes data handling easy and efficient.
After loading, the first few rows of the dataset are displayed to confirm that the data has been read correctly.

In [5]:
import pandas as pd  #import the library

df = pd.read_csv("train_loan.csv")   # Load the loan prediction dataset 

df.head() # Display first 5 rows to verify data

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


Understanding the Dataset

Before cleaning the data, it is important to understand what the dataset contains.
This step helps us identify the number of rows and columns, the data types of each feature, and whether there are any missing values in the dataset.

In [6]:
df.info()  #check dataset structure and data types

df.isnull().sum()   #check for missing values in each column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

Data Preprocessing

Missing Value Handling
Some columns in the dataset contain missing values.
If these values are not handled properly, they can affect the accuracy of the model.

In this case:
Missing values in categorical columns are replaced using the most frequent value (mode).
Missing values in numerical columns are replaced using the median, which is less affected by extreme values.

In [13]:
#fill missing values in categorical columns using mode

df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])


#fill missing values in numerical columns using median

df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].median())

In [12]:
df.isnull().sum() #checking is there any missing values

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

Outlier Detection

Outliers are values that are very different from the rest of the data.
These values can disturb the learning process of a machine learning model.
To handle outliers, the Interquartile Range (IQR) method is used, which helps in identifying and limiting extreme values in numerical columns.

In [15]:
df[['ApplicantIncome','CoapplicantIncome','LoanAmount']].describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount
count,614.000000,614.000000,614.000000
mean,4617.111564,1419.702231,137.365635
std,2479.851729,1624.605892,55.779749
min,150.000000,0.000000,9.000000
25%,2877.500000,0.000000,100.250000
50%,3812.500000,1188.500000,128.000000
75%,5795.000000,2297.250000,164.750000
max,10171.250000,5743.125000,261.500000


In [17]:
#detect and treat outliers using IQR method

Q1 = df[['ApplicantIncome','CoapplicantIncome','LoanAmount']].quantile(0.25)
Q3 = df[['ApplicantIncome','CoapplicantIncome','LoanAmount']].quantile(0.75)
IQR =Q3-Q1

df[['ApplicantIncome','CoapplicantIncome','LoanAmount']] =df[
    ['ApplicantIncome','CoapplicantIncome','LoanAmount']
].clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

In [19]:
df[['ApplicantIncome','CoapplicantIncome','LoanAmount']].describe()


,ApplicantIncome,CoapplicantIncome,LoanAmount
count,614.000000,614.000000,614.000000
mean,4617.111564,1419.702231,137.365635
std,2479.851729,1624.605892,55.779749
min,150.000000,0.000000,9.000000
25%,2877.500000,0.000000,100.250000
50%,3812.500000,1188.500000,128.000000
75%,5795.000000,2297.250000,164.750000
max,10171.250000,5743.125000,261.500000


ENCODING

Machine learning algorithms work only with numerical data. Since the dataset contains categorical values such as gender,marital status,and property area, these values are converted into numerical form using Label encoding

In [21]:
df.shape
#display dataset shape

(614, 13)

In [23]:

#import LabelEncoder class from sklearn preprocessing module
from sklearn.preprocessing import LabelEncoder

#create an object of LabelEncoder and convert categorical values into numerical form
le = LabelEncoder()
cat_cols = ['Gender','Married','Dependents','Education','Self_Employed','Property_Area','Loan_Status']

#loop through each categorical column
for col in cat_cols:
     #convert categorical values into numerical values for the current column
    df[col]=le.fit_transform(df[col])

In [22]:
df.dtypes #display the data types of dataset

Loan_ID               object
Gender                 int64
Married                int64
Dependents             int64
Education              int64
Self_Employed          int64
ApplicantIncome      float64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area          int64
Loan_Status            int64
dtype: object

SCALING

The numerical features in the dataset have different ranges.
Feature scaling is applied to bring all numerical values to a common scale so that no single feature dominates the model.
Here, StandardScaler is used to standardize the numerical features.

In [26]:
#import StandardScaler from sklearn preprocessing module
from sklearn.preprocessing import StandardScaler

#create an object of StandardScaler
scaler = StandardScaler()

#list of numerical columns that need to be scaled
num_cols = ['ApplicantIncome', 'CoapplicantIncome',
            'LoanAmount', 'Loan_Amount_Term']

#apply standard scaling (mean=0, standard deviation =1) to numerical columns
df[num_cols] = scaler.fit_transform(df[num_cols])


In [28]:
df[['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']].describe()
# Check summary statistics of scaled numerical columns

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term
count,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02
mean,-7.232723e-18,1.157236e-17,2.459126e-17,1.301890e-17
std,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00
min,-1.802831e+00,-8.745873e-01,-2.303171e+00,-5.132498e+00
25%,-7.020702e-01,-8.745873e-01,-6.659388e-01,2.732313e-01
50%,-3.247241e-01,-1.424288e-01,-1.680408e-01,2.732313e-01
75%,4.753707e-01,5.406008e-01,4.913377e-01,2.732313e-01
max,2.241532e+00,2.663383e+00,2.227252e+00,2.137276e+00


In [29]:
df[['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']].head()
# Display the first few rows of scaled numerical features to verify scaling

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term
0,0.497164,-0.874587,-0.168041,0.273231
1,-0.013767,0.054395,-0.168041,0.273231
2,-0.652632,-0.874587,-1.280462,0.273231
3,-0.820924,0.578025,-0.311579,0.273231
4,0.558104,-0.874587,0.065209,0.273231


FINAL PREPROCESSED DATASET

After completing all the preprocessing steps, the dataset becomes clean and well-structured.
It is now suitable for training machine learning models for loan prediction.

In [25]:
df.head() #view the final cleaned and processed dataset

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,1,0,0,0,0,0.497164,-0.874587,-0.168041,0.273231,1.0,2,1
1,LP001003,1,1,1,0,0,-0.013767,0.054395,-0.168041,0.273231,1.0,0,0
2,LP001005,1,1,0,0,1,-0.652632,-0.874587,-1.280462,0.273231,1.0,2,1
3,LP001006,1,1,0,1,0,-0.820924,0.578025,-0.311579,0.273231,1.0,2,1
4,LP001008,1,0,0,0,0,0.558104,-0.874587,0.065209,0.273231,1.0,2,1


CONCLUSION

In this case study, the loan prediction dataset was successfully preprocessed.
Missing values were handled, outliers were treated, categorical features were encoded, and numerical features were scaled.
The final dataset is now ready to be used for further analysis and model building.